## **Assignment 2 Group 63**
**Student Name:** Huzaifa Nissare-Houssen   
 **Student Number:** 300172186  
 **Student Name:** Fahmi Sajid Ahmed   
 **Student Number:** 300250180

 Huzaifa did Data Checker
 Fahmi did the missing data and imputation



### **Dataset Description: Netflix Movies and TV Shows**

---

#### **1. General Information**

 **Dataset Name:** Netflix Movies and TV Shows
   **Author:** Shivam Bansal   
   **Source:** https://www.kaggle.com/datasets/shivamb/netflix-shows  
   **Purpose:** This dataset was created to provide a comprehensive listing of all the movies and TV shows available on Netflix as of 2021. It is a real-world dataset intended for exploratory data analysis (EDA), trend tracking in the entertainment industry, and building recommendation engines. Researchers use it to compare content distribution across different countries and genres.

---

#### **2. Shape**

 **Rows:** 8,807  
 **Columns:** 12

---

#### **3. Features & Descriptions**

| Feature | Description | Type |
| --- | --- | --- |
| **`show_id`** | A unique identifier assigned to each movie or TV show. | **Categorical (ID)** |
| **`type`** | Identifies if the entry is a 'Movie' or a 'TV Show'. | **Categorical** |
| **`title`** | The official name of the content. | **Categorical (Text)** |
| **`director`** | The director(s) of the movie or show. (Contains many null values). | **Categorical** |
| **`cast`** | The primary actors and actresses featured in the title. | **Categorical** |
| **`country`** | The country or countries involved in the production of the content. | **Categorical** |
| **`date_added`** | The date the title was officially added to the Netflix library. | **Categorical (Date)** |
| **`release_year`** | The actual year the movie or show was first released. | **Numerical** |
| **`rating`** | The age-suitability rating (e.g., TV-MA, PG-13, R). | **Categorical** |
| **`duration`** | The length in minutes (Movies) or number of seasons (TV Shows). | **Numerical/Mixed** |
| **`listed_in`** | The genres assigned to the content (e.g., Horror, Documentaries). | **Categorical** |
| **`description`** | A short summary of the plot or premise. | **Categorical (Text)** |

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import random
import string

url = 'https://raw.githubusercontent.com/Fahmi-IT/CSI4142_A2/refs/heads/main/data/netflix_titles.csv'
clean_data = pd.read_csv(url)

clean_data.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:


def inject_numbers(df, column, decimal_percentage, min_val, max_val, type='int'):

    df_corrupted = df.copy()
    
    # 2. Calculate the number of rows to change
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        if type == 'float':
            random_values = np.random.uniform(min_val, max_val + 1, size=n_to_replace)
        
        
        df_corrupted.loc[indices_to_replace, column] = random_values
        
    return df_corrupted




def add_numeric_sort_key(df, source_column):

    df_result = df.copy()
    cleaned_series = df_result[source_column].astype(str).str.replace(r'[^0-9]', '', regex=True)
    cleaned_series = cleaned_series.replace('', pd.NA)
    df_result['sort_key'] = pd.to_numeric(cleaned_series, errors='coerce').astype('Int64')

    return df_result

def compare_heads(df1, df2, label1="Original", label2="Modified"):
    # This creates a side-by-side view using basic HTML/CSS
    html_str = f"""
    <div style="display: flex; gap: 50px;">
        <div>
            <h3>{label1}</h3>
            {df1.head(3).to_html()}
        </div>
        <div>
            <h3>{label2}</h3>
            {df2.head(3).to_html()}
        </div>
    </div>
    """
    display(HTML(html_str))

def get_column_differences(df1, df2, join_col, compare_col):
    """
    Joins two dataframes and returns a list of IDs where 
    the compare_col values are different.
    """
    # 1. Merge the dataframes
    merged = pd.merge(df1, df2, on=join_col, suffixes=('_df1', '_df2'))
    
    # 2. Identify the differences
    # Note: Use .ne() or != for comparison
    diff_mask = merged[f'{compare_col}_df1'] != merged[f'{compare_col}_df2']
    
    # 3. Extract the join_col (show_id) for those rows
    affected_ids = merged.loc[diff_mask, join_col].unique().tolist()
    
    return affected_ids

def generate_report(clean_data, dirty_data, column):
    potential_to_remove = ['description', 'cast', 'date_added', 'listed_in', 'sort_key']
    
    if column in potential_to_remove:
        potential_to_remove.remove(column)
    
    sorted_clean = add_numeric_sort_key(clean_data, 'show_id').sort_values(by='sort_key')
    sorted_dirty = add_numeric_sort_key(dirty_data, 'show_id').sort_values(by='sort_key')
    
    diff = get_column_differences(sorted_clean, sorted_dirty, 'show_id', column)
    sorted_clean = sorted_clean[sorted_clean['show_id'].isin(diff)].copy()
    sorted_dirty = sorted_dirty[sorted_dirty['show_id'].isin(diff)].copy()
    
    
    compare_heads(
        sorted_clean.drop(columns=potential_to_remove, errors='ignore'), 
        sorted_dirty.drop(columns=potential_to_remove, errors='ignore'),
        label1="Original",
        label2="Changed"
    )
    
    

### Data Type Validatity Checker

This test verfies that the data type of the enteries is of the expected type

In [100]:


def inject_type_errors(df, column, decimal_percentage=0.1, error_type='float'):
    df_result = df.copy()
    

    df_result[column] = df_result[column].astype(object)
    
    n_rows = len(df_result)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        target_indices = np.random.choice(df_result.index, size=n_to_replace, replace=False)
        
        if error_type == 'string':
            bad_data = "INVALID_TYPE"
        elif error_type == 'float':
            bad_data = 999.99  
        else:
            bad_data = None
            
        df_result.loc[target_indices, column] = bad_data
        
    return df_result


dirty_data = inject_type_errors(clean_data, 'release_year', 0.1, 'float')



In [101]:
def check_data_type(df, column, expected_type):
    errors = df[df[column].apply(lambda x: not isinstance(x, expected_type))]
    return errors

errors = check_data_type(dirty_data, 'release_year', int)



In [ ]:
compare_heads(clean_data, errors, 'Original', 'Changed')

### Range Validity Checker

This test verifies if the data is in a valid range. The range is the accepted values.

In [73]:
def inject_numbers(df, column, decimal_percentage, min_val, max_val):

    df_corrupted = df.copy()
    
    # 2. Calculate the number of rows to change
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        
        df_corrupted.loc[indices_to_replace, column] = random_values
        
    return df_corrupted

dirty_data = inject_numbers(clean_data,'release_year',0.1, 1765, 1890)

In [74]:
def check_range(df, column, min_val, max_val):
    return df[(df[column] < min_val) | (df[column] > max_val)]

errors = check_range(dirty_data, 'release_year', 1900, 2026)

In [ ]:
#final report with example
generate_report(clean_data, errors, 'release_year')

### Format Validity Checker

This test verifies if data is in a valid format.

In [93]:
def inject_string_errors(df, column, percentage, string_length=5):
    """
    Creates a copy of a dataframe and replaces a % of a column 
    with a random string (e.g., 'aZ3pQ').
    """
    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * (percentage / 100))
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        chars = string.ascii_letters + string.digits
        
        random_strings = [
            ''.join(random.choice(chars) for _ in range(string_length)) 
            for _ in range(n_to_replace)
        ]
        
        df_corrupted.loc[indices_to_replace, column] = random_strings
        
    return df_corrupted

dirty_data = inject_string_errors(clean_data, 'date_added', 0.1)

In [22]:
def check_format(df, column, regex_pattern):
    return df[~df[column].astype(str).str.contains(regex_pattern, na=True)]


valid_pattern = '^(January|February|March|April|May|June|July|August|September|October|November|December)\\s\\d{1,2},\\s\\d{4}$'

errors = check_format(dirty_data, 'date_added', valid_pattern).head()

/var/folders/3w/1_sy9lz53_d08dv59ywyyw700000gn/T/ipykernel_97771/2485409835.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return df[~df[column].astype(str).str.contains(regex_pattern, na=True)]


In [ ]:
#final report with example
generate_report(clean_data, errors, 'date_added')

### Consistency Validity Checker
 This test checks the consistency of two rows. So that the data across rows is consistent

In [85]:

def invalidate_date_added(df, column='date_added', decimal_percentage=0.1):
  
    df_result = df.copy()
    

    n_rows = len(df_result)
    n_to_invalidate = int(n_rows * decimal_percentage)
    
    if n_to_invalidate > 0:
        target_indices = np.random.choice(df_result.index, size=n_to_invalidate, replace=False)
        
        
        subset = df_result.loc[target_indices, column].str.split(',', n=1, expand=True)
        
        if subset.shape[1] >= 2:
            numeric_part = pd.to_numeric(subset[1].str.strip(), errors='coerce')
            modified_number = (numeric_part - 100).astype(str)
            
       
            df_result.loc[target_indices, column] = subset[0] + ', ' + modified_number
            
    return df_result
dirty_data = invalidate_date_added(clean_data)

In [86]:
def year_consistency(df):
    df['date_added'] =  pd.to_numeric(df['date_added'].str.split(',', expand=True)[1], errors='coerce')
    mask = df['date_added'] < df['release_year']
    return df[mask]


errors = year_consistency(dirty_data)


In [87]:
#final report with example
generate_report(clean_data, errors, 'date_added')

[4, 27, 29, 37, 41, 64, 82, 88, 99, 103, 121, 123, 129, 161, 170, 177, 192, 215, 221, 230, 260, 264, 265, 266, 267, 270, 276, 322, 337, 372, 373, 376, 391, 401, 414, 421, 422, 430, 435, 455, 462, 470, 475, 479, 485, 491, 503, 511, 517, 520, 527, 531, 536, 560, 562, 571, 580, 591, 596, 599, 616, 649, 659, 671, 678, 679, 687, 698, 710, 711, 727, 728, 730, 739, 749, 755, 772, 778, 796, 827, 853, 866, 868, 874, 879, 896, 897, 921, 922, 923, 924, 960, 964, 965, 973, 979, 981, 985, 992, 1013, 1014, 1035, 1036, 1042, 1043, 1050, 1055, 1057, 1059, 1069, 1073, 1078, 1102, 1112, 1115, 1123, 1128, 1142, 1144, 1145, 1146, 1170, 1172, 1198, 1204, 1209, 1215, 1217, 1225, 1234, 1250, 1258, 1281, 1290, 1302, 1311, 1312, 1314, 1324, 1329, 1341, 1347, 1352, 1364, 1368, 1369, 1379, 1381, 1383, 1393, 1396, 1408, 1430, 1432, 1446, 1463, 1468, 1486, 1514, 1530, 1542, 1544, 1552, 1557, 1597, 1607, 1608, 1624, 1629, 1631, 1644, 1692, 1696, 1697, 1729, 1742, 1751, 1755, 1767, 1768, 1780, 1791, 1829, 1831, 1837

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
3,4,TV Show,Jailbirds New Orleans,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV"
26,27,Movie,Minsara Kanavu,Rajiv Menon,NaN,"September 21, 2021",1997,TV-PG,147 min,"Comedies, International Movies, Music & Musicals"
28,29,Movie,Dark Skies,Scott Stewart,United States,"September 19, 2021",2013,PG-13,97 min,"Horror Movies, Sci-Fi & Fantasy"
,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
3,4,TV Show,Jailbirds New Orleans,NaN,NaN,1921.0,2021,TV-MA,1 Season,"Docuseries, Reality TV"
26,27,Movie,Minsara Kanavu,Rajiv Menon,NaN,1921.0,1997,TV-PG,147 min,"Comedies, International Movies, Music & Musicals"
28,29,Movie,Dark Skies,Scott Stewart,United States,1921.0,2013,PG-13,97 min,"Horror Movies, Sci-Fi & Fantasy"


### Uniqueness Validity Checker

This test validates that data is unique and that there are no repeats


In [16]:
def inject_ids(df, column, decimal_percentage, min_val, max_val):

    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        random_values_with_s = "s" + random_values.astype(str)
        
        df_corrupted.loc[indices_to_replace, column] = random_values_with_s
        
    return df_corrupted


dirty_data = inject_ids(clean_data, "show_id", 0.05, 1, 1000)

In [17]:
def check_uniqueness(df, column):
    return df[df.duplicated(subset=[column], keep=False)]


errors = check_uniqueness(dirty_data, 'show_id')
errors.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
12,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic","September 23, 2021",2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...
14,s15,TV Show,Crime Stories: India Detectives,NaN,NaN,NaN,"September 22, 2021",2021,TV-MA,1 Season,"British TV Shows, Crime TV Shows, Docuseries",Cameras following Bengaluru police on the job ...


In [20]:

def get_common_rows(df1, df2, column):
    common_values = set(df2[column])
    common_df = df1[df1[column].isin(common_values)].copy()
    
    return common_df
    
    
common_values = get_common_rows(clean_data, errors, 'show_id')
compare_heads(
        common_values, 
       clean_data,
        label1="Original",
        label2="Changed"
    )

    

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, Sofia Carson, Liza Koshy, Ken Jeong, Elizabeth Perkins, Jane Krakowski, Michael McKean, Phil LaMarr",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,"Equestria's divided. But a bright-eyed hero believes Earth Ponies, Pegasi and Unicorns should be pals — and, hoof to heart, she’s determined to prove it."
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra Duah, Nick Medley, Mutabaruka, Afemo Omilami, Reggie Carter, Mzuri","United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model slips back in time, becomes enslaved on a plantation and bears witness to the agony of her ancestral past."
,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."


### Presecence Validity Checker

This test validates data that is not null or empty

In [53]:
#introduce error
def inject_presence_errors(df, column, decimal_percentage):

    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_nullify = int(n_rows * decimal_percentage)
    
    if n_to_nullify > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_nullify, replace=False)
        
     
        df_corrupted.loc[indices_to_replace, column] = np.nan
        
    return df_corrupted

dirty_data = inject_presence_errors(clean_data, 'description', 0.1)

In [54]:
def check_presence(df, column):
    return df[df[column].isna() | (df[column].astype(str).str.strip() == "")]

errors = check_presence(dirty_data, 'description')

In [ ]:
#final report
generate_report(clean_data, errors, 'description')

### Length Validity Checker

This test tests if data is a certian length

In [81]:
#introduce error
dirty_data = inject_string_errors(clean_data, 'director', 0.1, 50)

In [82]:
def check_length(df, column, min_len, max_len):
    lengths = df[column].astype(str).str.len()
    return df[(lengths < min_len) | (lengths > max_len)]

errors = check_length(dirty_data,'director', 1, 30)

In [ ]:
generate_report(clean_data, errors, 'director')

### Look-up Validity Checker
This test verifies that all that elements are part of a valid set


In [ ]:
dirty_data = inject_string_errors(clean_data, 'rating', 0.1)

In [ ]:

def get_invalid_tv_ratings(df, column='rating'):

  
    valid_ratings = {
        'TV-Y', 'TV-Y7', 'TV-G', 'TV-PG', 'TV-14', 'TV-MA', 
        'G', 'PG', 'PG-13', 'R', 'NC-17', 'NR', 'UR'
    }
    

    check_series = df[column].astype(str).str.strip().str.upper()
    

    invalid_mask = ~check_series.isin(valid_ratings)
    
    return df[invalid_mask].copy()

errors = get_invalid_tv_ratings(dirty_data)

In [ ]:
generate_report(clean_data, errors, 'ratings')

Group Number: 63

Name: Fahmi Sajid Ahmed

Student Number: 300250180

In [1]:
# Import Libraries + required assets

import pandas as pd;
import numpy as np;
from sklearn.preprocessing import StandardScaler;
from sklearn.impute import KNNImputer;
from sklearn.linear_model import LinearRegression;

# Dataset
url = "https://raw.githubusercontent.com/Fahmi-IT/CSI4142_A2/refs/heads/a2_part2/global_tech_fundamentals_2026.csv";
df = pd.read_csv(url);

numeric_cols = [
    "Market_Cap_B", "Employees", "Total_Revenue_B", "Research_And_Development_B",
    "Net_Income_B", "Total_Debt_B", "Free_Cash_Flow_B", "R&D_to_Revenue_Pct",
    "Gross_Margin_Pct", "Operating_Margin_Pct", "Profit_Margin_Pct",
    "Trailing_PE", "Forward_PE", "Price_to_Book", "Beta_Volatility",
    "Institutional_Ownership_Pct", "Short_Float_Pct", "Dividend_Yield_Pct",
    "52_Week_Change_Pct"
];

# Set seed for reproducibility
RANDOM_SEED = 7;
np.random.seed(RANDOM_SEED);

Dataset URL: https://www.kaggle.com/datasets/kanchana1990/global-tech-giants-fundamentals-and-valuations-2026

This is CSI4142 A2 Part 2. The purpose of this portion of the assignment is to explore various imputation methods.

The Global Tech Giants: Fundamentals & Valuations 2026 dataset describes a deep cross-sectional snapshot of the core business mechanics of the top 100+ tech, semiconductor, and software companies in the world. The author, Kanchana Gajamuthu, engineered the dataset specifically for Exploratory Data Analysis so that data scientists and AI engineers alike could use it (leaning towards training purposes).

The dataset consists of 22 attributes/columns and 103 different data entries/rows.

The attributes are as follows:
- **Ticker**: (Categorical) The stock ticker symbol used to identify the company on public exchanges.
- **Company_Name**: (Categorical) The full legal name of the company.
- **Industry**: (Categorical) The primary industry sector in which the company operates.
- **Market_Cap_B**: (Numerical) The company’s total market capitalization in billions of dollars.
- **Employees**: (Numerical) The total number of employees working for the company.
- **Total_Revenue_B**: (Numerical) The company’s total annual revenue in billions of dollars.
- **Research_And_Development_B**: (Numerical) The annual amount spent on research and development in billions of dollars.
- **Net_Income_B**: (Numerical) The company’s net profit after expenses in billions of dollars.
- **Total_Debt_B**: (Numerical) The total outstanding debt held by the company in billions of dollars.
- **Free_Cash_Flow_B**: (Numerical) The company’s free cash flow in billions of dollars.
- **R&D_to_Revenue_Pct**: (Numerical) The percentage of revenue allocated to research and development.
- **Gross_Margin_Pct**: (Numerical) The percentage of revenue remaining after cost of goods sold.
- **Operating_Margin_Pct**: (Numerical) The percentage of revenue remaining after operating expenses.
- **Profit_Margin_Pct**: (Numerical) The percentage of revenue that translates into net profit.
- **Trailing_PE**: (Numerical) The price-to-earnings ratio based on past (historical) earnings.
- **Forward_PE**: (Numerical) The price-to-earnings ratio based on projected future earnings.
- **Price_to_Book**: (Numerical) The ratio of the company’s market value to its book value.
- **Beta_Volatility**: (Numerical) A measure of the stock’s volatility relative to the overall market.
- **Institutional_Ownership_Pct**: (Numerical) The percentage of shares owned by institutional investors.
- **Short_Float_Pct**: (Numerical) The percentage of shares available for trading that are currently sold short.
- **Dividend_Yield_Pct**: (Numerical) The annual dividend yield expressed as a percentage of the stock price.
- **52_Week_Change_Pct**: (Numerical) The percentage change in the stock price over the past 52 weeks.

1.1

**Type of Imputation**: Default Value Imputation (Median)

**Which attribute is being impacted**: Free_Cash_Flow_B

**What is Default Value Imputation (Median)?**

Default Value Imputation (Median) is an imputation method that involves replacing missing values with the median value of that attribute in the dataset.

1.2

The column Free_Cash_Flow_B was chosen to undergo MCAR removal because it is a continuous financial metric that tends to be right-skewed due to the tech giant outliers (Apple, Microsoft, etc.), which makes the median a better default than, say, the mean. A predetermined number of values were removed at random using the np.random.choice() function and replacing the selected indices with NaN.

In [2]:
# 1.3
TARGET = "Free_Cash_Flow_B";

MISSING_RATE = 0.15;  # 15% of values will be removed (MCAR)

# Remove values according to MCAR method (replace with NaN)
df_mcar = df.copy();
missing_indices = np.random.choice(df_mcar.index, size=int(len(df_mcar) * MISSING_RATE), replace=False);
df_mcar.loc[missing_indices, TARGET] = np.nan;

# Default Value Imputation (Median)
median_value = df_mcar[TARGET].median();
print(f"Computed median  : {median_value:.4f}");
df_imputed = df_mcar.copy();
df_imputed[TARGET] = df_imputed[TARGET].fillna(median_value);

# Summary of Changes
print(f"\nStat Summary");
summary = pd.DataFrame({
    "Original"  : df[TARGET].describe(),
    "After MCAR": df_mcar[TARGET].describe(),
    "Imputed"   : df_imputed[TARGET].describe(),
});
print(summary.to_string());

# Save MCAR removed + median imputed versions to file
df_mcar.to_csv("tech_mcar.csv", index=False);
df_imputed.to_csv("tech_median_imputed.csv", index=False);
print("\nFiles saved: tech_mcar.csv, tech_median_imputed.csv");

Computed median  : 1.4150

Stat Summary
         Original  After MCAR     Imputed
count  103.000000   88.000000  103.000000
mean    20.528835   23.403182   20.201019
std    100.330298  108.348644  100.368260
min     -4.950000   -4.950000   -4.950000
25%      0.590000    0.692500    0.755000
50%      1.360000    1.415000    1.415000
75%      6.410000    6.245000    5.030000
max    992.380000  992.380000  992.380000

Files saved: tech_mcar.csv, tech_median_imputed.csv


1.4

The Default Value Imputation method using the median filled 15 values using the median value of 1.415B. The Mean Absolute Error (MAE) was 9.1 with a Root Mean Squared Error (RMSE) of 25.08, with a slightly negative R^2 score of -0.13. In comparison to a separate analysis (not displayed here) when using the mean, the median falls short. However, this is largely due to the fact that the Free_Cash_Flow_B attribute has a very high standard deviation of 100.33 due to the outliers on the high end (Microsoft, Apple, etc.). This causes the median to be a poor representation for values far from the centre. Despite this, the aggregate distribution was well-preserved, only slightly impacting the mean and the standard deviation barely budging to 100.37. This confirms that the Default Value Imputation via the median is conservative and distribution-stable, even when individual predictions are off the mark.

2.1

**Type of Imputation**: Correlation Imputation

**Which attribute is being impacted**: Employees

**What is Correlation Imputation?**

Correlation Imputation is an imputation method that estimates missing values via linear regression with another numerical feature (in this case, Total_Revenue_B) (larger revenue = more employees).

2.2

MAR removal was used on the Employees attribute as hardware, consumer electronics, and internet retail companies are given much higher rates of missing values in comparison to software companies, which reflects a realistic MAR scenario. This was done by assigning each industry a specific probability of having its Employees value removed (ranging from 80% for Computer Hardware to 5% for Software - Application) and then randomly selecting a subset of those rows proportional to the assigned probability set to NaN.

In [3]:
# 2.3
TARGET = "Employees";

# MAR Removal
industry_missing_prob = {
    "Computer Hardware": 0.80,
    "Consumer Electronics": 0.80,
    "Internet Retail": 0.60,
    "Semiconductor Equipment & Materials": 0.50,
    "Electronic Components": 0.50,
    "Semiconductors": 0.30,
    "Information Technology Services": 0.20,
    "Software - Application": 0.05,
    "Software - Infrastructure": 0.05,
    "Internet Content & Information": 0.05,
    "Credit Services": 0.05,
    "Entertainment": 0.05,
    "Communication Equipment": 0.05,
    "Travel Services": 0.05,
    "Electronic Gaming & Multimedia": 0.05,
    "Specialty Business Services": 0.05,
    "Leisure": 0.05,
};

df_mar = df.copy();
missing_mask = pd.Series(False, index=df_mar.index);

for industry, prob in industry_missing_prob.items():
    idx = df_mar[df_mar["Industry"] == industry].index;
    if len(idx) == 0:
        continue;
    n_remove = max(1, int(len(idx) * prob)) if prob >= (1 / len(idx)) else 0;
    chosen = np.random.choice(idx, size=min(n_remove, len(idx)), replace=False);
    missing_mask.loc[chosen] = True;

df_mar.loc[missing_mask, TARGET] = np.nan;

# Verify MAR Property
print(f"\nMissing rate by Industry:");
mar_check = (
    df_mar.groupby("Industry")[TARGET]
    .apply(lambda x: f"{x.isna().mean() * 100:.0f}%  ({x.isna().sum()}/{len(x)})")
    .sort_values(ascending=False)
);
print(mar_check.to_string());

PREDICTOR = "Total_Revenue_B";
complete_mask = df_mar[TARGET].notna();
df_complete = df_mar[complete_mask];
df_missing = df_mar[~complete_mask];
X_train = df_complete[[PREDICTOR]].values;
y_train = df_complete[TARGET].values;
model = LinearRegression();
model.fit(X_train, y_train);

X_missing = df_missing[[PREDICTOR]].values;
predictions = np.maximum(model.predict(X_missing), 0);  # employees can't be negative
df_imputed = df_mar.copy();
df_imputed.loc[~complete_mask, TARGET] = predictions.round().astype(int);

print(f"\nStats Summary");
summary = pd.DataFrame({
    "Original": df[TARGET].describe(),
    "After MAR": df_mar[TARGET].describe(),
    "Imputed": df_imputed[TARGET].describe(),
});
print(summary.to_string());

original_at_missing = df.loc[~complete_mask, TARGET];
imputed_at_missing = df_imputed.loc[~complete_mask, TARGET];

mae = np.mean(np.abs(original_at_missing - imputed_at_missing));
rmse = np.sqrt(np.mean((original_at_missing - imputed_at_missing) ** 2));
print(f"\nImputation Quality (vs ground truth)");
print(f"MAE: {mae:,.0f} employees");
print(f"RMSE: {rmse:,.0f} employees");

# Save MAR removed + correlation imputed versions to file
df_mar.to_csv("tech_mar.csv", index=False);
df_imputed.to_csv("tech_correlation_imputed.csv", index=False);
print("\nFiles saved: tech_mar.csv, tech_correlation_imputed.csv");


Missing rate by Industry:
Industry
Internet Retail                         50%  (4/8)
Computer Hardware                       50%  (1/2)
Software - Application                  5%  (1/21)
Semiconductor Equipment & Materials     40%  (2/5)
Semiconductors                         29%  (5/17)
Information Technology Services         12%  (1/8)
Internet Content & Information           0%  (0/9)
Credit Services                          0%  (0/4)
Communication Equipment                  0%  (0/2)
Entertainment                            0%  (0/2)
Software - Infrastructure               0%  (0/19)
Leisure                                  0%  (0/1)
Electronic Gaming & Multimedia           0%  (0/1)
Electronic Components                    0%  (0/1)
Consumer Electronics                     0%  (0/1)
Specialty Business Services              0%  (0/1)
Travel Services                          0%  (0/1)

Stats Summary
           Original      After MAR        Imputed
count  1.030000e+02      89.0000

2.4

The correlation imputation on the Employees attribute (using Total_Revenue_B for the correlation via linear regression) imputed 14 missing employee counts, producing an MAE of 240,716 employees and an RMSE of 512,599. The large absolute errors are to be expected due to the attribute's standard deviation of 196,026. The imputed dataset mean strayed far from the original, and the standard deviation compressed a lot (from 196,026 to 55,062.72), an important side effect of regression imputation.

The discrepancy between the removal rate and the intended probabilities is due to the small group sizes.

3.1

**Type of Imputation**: Similarity-based Imputation

**Which attribute is being impacted**: Dividend_Yield_Pct

**What is Similarity-based Imputation?**

Similarity-based imputation basically uses the similarity between records in the dataset to estimate missing ones. A common application of this is kNN.

3.2

MNAR removal was applied to the Dividend_Yield_Pct attribute because the missingness can be tied directly to the value itself in 3 tiers: zero-dividend companies, low-yield companies, and high-yield companies. This helps represent a realistic MNAR scenario (those with nothing good to show won't show it). This was done by essentially generating a random continuous value between 0 and 1. If the generated value fell below the threshold according to the row's tier classification, then the attribute value for that row would be set to NaN.

In [4]:
# 3.3
TARGET = "Dividend_Yield_Pct";

df_mnar = df.copy();
missing_mask = pd.Series(False, index=df_mnar.index);
for idx, row in df_mnar.iterrows():
    val = row[TARGET];
    if val == 0:
        prob = 0.85;
    elif val <= 50:
        prob = 0.40;
    else:
        prob = 0.05;
    if np.random.rand() < prob:
        missing_mask.loc[idx] = True;
df_mnar.loc[missing_mask, TARGET] = np.nan;

print(f"\nMissing rate by value tier (ground truth):");
bins = [-1, 0, 50, float("inf")];
labels = ["Zero (0)", "Low (1–50)", "High (>50)"];
tiers = pd.cut(df[TARGET], bins=bins, labels=labels);
mnar_check = pd.DataFrame({"tier": tiers, "is_missing": missing_mask});
print(mnar_check.groupby("tier", observed=True)["is_missing"]
      .apply(lambda x: f"{x.mean()*100:.0f}%  ({x.sum()}/{len(x)})")
      .to_string());

KNN_FEATURES = [
    "Gross_Margin_Pct",
    "Forward_PE",
    "Beta_Volatility",
    "Operating_Margin_Pct",
    TARGET
];
K = 5  # number of neighbours
scaler = StandardScaler();
knn_data = df_mnar[KNN_FEATURES].copy();
feature_cols = KNN_FEATURES[:-1];

knn_data[feature_cols] = scaler.fit_transform(knn_data[feature_cols]);

# Run KNN imputer
imputer = KNNImputer(n_neighbors=K, weights="distance");
imputed_arr = imputer.fit_transform(knn_data);

# Rebuild dataframe and reverse standardisation on features (TARGET was not scaled)
imputed_knn = pd.DataFrame(imputed_arr, columns=KNN_FEATURES, index=df_mnar.index);

df_imputed = df_mnar.copy();
df_imputed[TARGET] = imputed_knn[TARGET];

print(f"\nStats Summary");
summary = pd.DataFrame({
    "Original": df[TARGET].describe(),
    "After MNAR": df_mnar[TARGET].describe(),
    "Imputed": df_imputed[TARGET].describe(),
});
print(summary.to_string());

original_at_missing = df.loc[missing_mask, TARGET];
imputed_at_missing = df_imputed.loc[missing_mask, TARGET];

mae = np.mean(np.abs(original_at_missing - imputed_at_missing));
rmse = np.sqrt(np.mean((original_at_missing - imputed_at_missing) ** 2));
print(f"\nImputation Quality (vs ground truth)");
print(f"MAE: {mae:.4f}");
print(f"RMSE: {rmse:.4f}");

# Save MNAR removed + similarity imputed versions to file
df_mnar.to_csv("tech_mnar.csv", index=False);
df_imputed.to_csv("tech_knn_imputed.csv", index=False);
print("\nFiles saved: tech_mnar.csv, tech_knn_imputed.csv");


Missing rate by value tier (ground truth):
tier
Zero (0)      91%  (50/55)
Low (1–50)     18%  (2/11)
High (>50)      5%  (2/37)

Stats Summary
         Original  After MNAR     Imputed
count  103.000000   49.000000  103.000000
mean    64.533981  129.530612  124.732182
std    100.759062  111.257603   87.827703
min      0.000000    0.000000    0.000000
25%      0.000000   49.000000   64.210769
50%      0.000000   93.000000  113.997463
75%     91.500000  210.000000  157.969202
max    477.000000  477.000000  477.000000

Imputation Quality (vs ground truth)
MAE: 115.3431
RMSE: 130.7634

Files saved: tech_mnar.csv, tech_knn_imputed.csv


3.4

The KNN Imputatation on the Dividend_Yield_Pct attribute (using K = 5) imputed a total of 54 values. Of the 54 that were imputed, 50 of them were originally 0s (non-dividend-paying companies) which got an average imputed value of 95.04 (which is much greater than their original value of 0). This produced an MAE of 115.3431 and an RMSE of 130.7634. This is mainly due to the MNAR removal method as the neighbors of non-dividend-paying companies in the KNN feature space may well be dividend-paying companies with high yields, giving the imputer no basis to correctly recover the true 0 (which inflated the mean).

References

Scikit-learn KNN Imputer Documentation 
(https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)

MCAR, MAR, MNAR Removal 
(https://uottawa.brightspace.com/d2l/le/content/567749/viewContent/7420050/View)